# einops-rearrange — ex7: NHWC ↔ NCHW round-trip with imshow

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-rearrange`. Running the final beacon cell reports progress against the `Einops: Rearrange` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Rearrange` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-rearrange`** (exercise 7). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-rearrange"
DD_SUBTOPIC = "Einops: Rearrange"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.rearrange — quick refresher

`rearrange(tensor, pattern, **axes_lengths)` is one operator with three jobs: **reorder** axes (`'h w -> w h'`), **compose** them (`'h w c -> (h w) c'`), and **decompose** them (`'(b1 b2) c -> b1 b2 c'`, with `b1=` or `b2=`). Every identifier on the right must appear on the left and vice versa.

The exercises below build on that: each one runs `rearrange` inside a small pipeline where you have to *see* what the layout did — by plotting it, by printing the shape at each step, or by combining 2–3 patterns into a single ML-adjacent transformation.

### Exercise 7 — NHWC ↔ NCHW round-trip with imshow

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Convert an image batch between NHWC (TF / matplotlib convention) and NCHW (PyTorch convention) using rearrange, and visually verify the round-trip preserves the image.
> Keywords: visualization, matplotlib, image-layout, axis-swap
> ```

**KCs targeted:** `rearrange-axis-swap`

PyTorch wants images as `(N, C, H, W)`. matplotlib's `imshow` wants them as `(H, W, C)`. Frameworks like TensorFlow keep `(N, H, W, C)`. Converting between them is one of the most common rearrange applications in real ML code — and getting it wrong silently corrupts your batch (the loss won't crash, the image just won't *look* like an image to the model).

Implement `ex7_nhwc_to_nchw_and_back(img_nhwc)`:
1. Take an image batch in NHWC layout, shape `(N, H, W, 3)`.
2. Convert it to NCHW (`(N, 3, H, W)`) — that's `chw`.
3. Convert `chw` back to NHWC — that's `roundtrip`.
4. Use matplotlib to plot the **first** image of `img_nhwc` and the first image of `roundtrip` side by side, with titles 'original' and 'after NHWC→NCHW→NHWC'.
5. Return the tuple `(chw, roundtrip)`.

If the two imshow panels don't look identical, you've axis-swapped wrong — that's the visual sanity check this exercise exists for.

In [ ]:
def ex7_nhwc_to_nchw_and_back(img_nhwc: Tensor) -> tuple[Tensor, Tensor]:
    import matplotlib.pyplot as plt
    chw = rearrange(img_nhwc, 'n h w c -> n c h w')
    roundtrip = rearrange(chw, 'n c h w -> n h w c')
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(img_nhwc[0].cpu().numpy())
    axes[0].set_title('original')
    axes[0].axis('off')
    axes[1].imshow(roundtrip[0].cpu().numpy())
    axes[1].set_title('after NHWC→NCHW→NHWC')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    return chw, roundtrip


<details><summary>Solution</summary>

```python
def ex7_nhwc_to_nchw_and_back(img_nhwc: Tensor) -> tuple[Tensor, Tensor]:
    import matplotlib.pyplot as plt
    chw = rearrange(img_nhwc, 'n h w c -> n c h w')
    roundtrip = rearrange(chw, 'n c h w -> n h w c')
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(img_nhwc[0].cpu().numpy())
    axes[0].set_title('original')
    axes[0].axis('off')
    axes[1].imshow(roundtrip[0].cpu().numpy())
    axes[1].set_title('after NHWC→NCHW→NHWC')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()
    return chw, roundtrip
```

**Why this isn't a `.permute` review.** Naming the axes (`n h w c -> n c h w`) makes the *meaning* of the transformation visible — `permute(0, 3, 1, 2)` is the same op but you have to decode three integers to know what moved where. In a real ML pipeline with 4–5 axes, that matters.

**Why imshow?** A wrong channel swap (e.g. `'n h w c -> n h c w'`) passes shape asserts but produces nonsense images. The visual sanity check is faster than chasing a silent accuracy regression.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex7',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()